# Fine-tuning DistilBERT with the Hugging Face Trainer API

This notebook fine-tunes `distilbert-base-uncased` to classify emotion in text using a reproducible 2,000-example subset of the Emotion dataset. It prints the dataset size and label distribution, creates a stratified training/validation split, and evaluates accuracy and weighted F1 after every epoch.

Install the dependencies from the repository root with `pip install -r Ai_task15/requirements.txt`. A GPU is recommended; CPU training is supported but may take considerably longer.

## 1. Imports and hardware check

In [1]:
from collections import Counter

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

print(f"CUDA GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA GPU available: False


## 2. Load a small subset and inspect its labels

The `split` configuration has a separate training split. We select at most 2,000 examples from that split and create our own stratified validation holdout, keeping the original test split out of training.

In [2]:
MAX_SAMPLES = 2_000
SEED = 42

dataset = load_dataset("dair-ai/emotion", "split", split="train")
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
label_names = dataset.features["label"].names

label_distribution = Counter(label_names[label] for label in dataset["label"])
print(f"Dataset size: {len(dataset):,} examples")
print("Label distribution:")
for label_name in label_names:
    print(f"  {label_name}: {label_distribution[label_name]:,}")

dataset_splits = dataset.train_test_split(
    test_size=0.2,
    seed=SEED,
    stratify_by_column="label",
)
dataset_splits["validation"] = dataset_splits.pop("test")
print(f"\nTraining examples: {len(dataset_splits['train']):,}")
print(f"Validation examples: {len(dataset_splits['validation']):,}")

Dataset size: 2,000 examples
Label distribution:
  sadness: 544
  joy: 702
  love: 173
  anger: 268
  fear: 236
  surprise: 77

Training examples: 1,600
Validation examples: 400


C:\Users\Admin\Desktop\Work\AI Task\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Admin\Desktop\Work\AI Task\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning

## 3. Tokenize in batches

`map(tokenize, batched=True)` applies the DistilBERT tokenizer efficiently to each dataset split.

In [3]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_splits = dataset_splits.map(tokenize, batched=True)
tokenized_splits = tokenized_splits.rename_column("label", "labels")
tokenized_splits

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 400
    })
})


Map: 100%|##########| 400/400 [00:00<00:00, 15997.35 examples/s]


## 4. Load the classification model

The label mappings keep model predictions interpretable as emotion names.

In [4]:
num_labels = len(label_names)
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)
print(f"Number of labels: {num_labels}")

Number of labels: 6


C:\Users\Admin\Desktop\Work\AI Task\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|##########| 100/100 [00:00<00:00, 400.76it/s]


## 5. Define accuracy and F1 metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

## 6. Configure and run training

Training evaluates and saves a checkpoint at the end of each epoch, then restores the checkpoint with the best validation weighted F1.

In [6]:
training_args = TrainingArguments(
    output_dir="./results/distilbert-emotion",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    weight_decay=0.01,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    eval_dataset=tokenized_splits["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)
evaluation_result = trainer.evaluate()
print("Final validation metrics:", evaluation_result)

{'eval_loss': '1.171', 'eval_accuracy': '0.5525', 'eval_f1': '0.4381', 'eval_runtime': '16.18', 'eval_samples_per_second': '24.73', 'eval_steps_per_second': '1.545', 'epoch': '1'}
{'eval_loss': '0.8596', 'eval_accuracy': '0.735', 'eval_f1': '0.6844', 'eval_runtime': '16.18', 'eval_samples_per_second': '24.73', 'eval_steps_per_second': '1.545', 'epoch': '2'}
{'eval_loss': '0.7616', 'eval_accuracy': '0.75', 'eval_f1': '0.6984', 'eval_runtime': '16.61', 'eval_samples_per_second': '24.09', 'eval_steps_per_second': '1.506', 'epoch': '3'}
{'train_runtime': '792.1', 'train_samples_per_second': '6.06', 'train_steps_per_second': '0.379', 'train_loss': '1.009', 'epoch': '3'}
TrainOutput(global_step=300, training_loss=1.0088398234049478, metrics={'train_runtime': 792.0775, 'train_samples_per_second': 6.06, 'train_steps_per_second': 0.379, 'total_flos': 58447129433472.0, 'train_loss': 1.0088398234049478, 'epoch': 3.0})
Final validation metrics: {'eval_loss': 0.7615965008735657, 'eval_accuracy': 0.

  0%|          | 0/300 [00:00<?, ?it/s]C:\Users\Admin\Desktop\Work\AI Task\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 96%|#########6| 24/25 [00:14<00:00,  1.77it/s]
                                                 
100%|##########| 25/25 [00:15<00:00,  1.91it/s]
                                               
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  1.43it/s]
C:\Users\Admin\Desktop\Work\AI Task\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 96%|#########6| 24/25 [00:15<00:00,  1.51it/s]
                                                 
100%|##########| 25/25 [00:15<00:00,  1.63it/s]
                                               
Writing model shard